# 🚧 Module 3 — Class 5: Build a Pipeline (and handle imbalance)

**Lesson:** [bepro-aiml.github.io/aiml-platform/#/module/3/class/5](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/5)

---

## 📖 Today's story

Look at your code from Classes 1-4. It's spread across 4 notebooks. Your manager says:

> *"OK, we want this in production tomorrow. When a NEW order arrives, run the same cleaning + scaling on it and predict. Show me the code."*

You panic. Your code is in 4 notebooks with 30 cells each. You can't just paste them into the production server.

**Today we wrap everything in a single `Pipeline` object.** One `.pkl` file = production-ready preprocessing.

Plus we'll cover the OTHER big issue lurking on Olist: only 7% of orders are late. **The data is imbalanced.** If we don't handle it, the model just predicts "on time" for everything and looks like 93% accurate — useless.

---

## 💡 What is a pipeline?

A **Pipeline** = a chain of transformations + final model, all bundled into one object.

Instead of:
```python
X_clean = clean(X)
X_imputed = impute(X_clean)
X_scaled = scaler.fit_transform(X_imputed)
X_encoded = encoder.fit_transform(X_scaled)
model.fit(X_encoded, y)
```

You write:
```python
pipeline.fit(X, y)
```

And in production: `pipeline.predict(new_order)` does ALL the steps in order automatically.

### 🚂 Train-track analogy

Imagine a train going through 5 stations. Each station does one job: cleaning, imputing, scaling, encoding, predicting.

Without a pipeline, you have to manually move the train between stations. Slow. Error-prone.

With a pipeline, the train rolls automatically through all 5 stations. Fast. Reproducible. Production-ready.

---

## 🎯 Today's plan

1. **Why pipelines** — leakage prevention + production reproducibility
2. **Build a `ColumnTransformer`** (different transforms for numeric vs categorical)
3. **Wrap it in a `Pipeline`** with a Logistic Regression
4. **Train + evaluate** end-to-end
5. **Imbalance handling** — class_weight='balanced' and SMOTE
6. **Save** the pipeline as `olist_preprocessor.pkl` for M4-M8

---

## 🚨 Section 1: The two killers a pipeline prevents

### 1️⃣ Data leakage

**Wrong:** fit a scaler on the FULL dataset, then split.

Why wrong? The scaler learned the test data's mean/std. The model now has a tiny advantage at test time. Production accuracy will be lower than test accuracy → silent failure.

**Right:** split first, fit ONLY on training.

A pipeline forces this — it can't peek at test data.

### 2️⃣ Train/serve skew

**Wrong:** clean data manually for training, then write different cleaning code for production.

Why wrong? Tiny inconsistencies between training-time and serve-time cleaning create silent bugs. Model accuracy drops in production for no obvious reason.

**Right:** ONE pipeline, ONE serialised object. Use the same `pipeline.predict()` everywhere.

### 🍞 Cooking analogy

Imagine you're a chef teaching an apprentice. You write down a recipe with exact measurements. The apprentice follows the recipe exactly.

Without a written recipe, every dish tastes slightly different. With it, every dish is reproducible.

**Pipeline = the written recipe for your data.**

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

df = pd.read_parquet('orders_step3.parquet').dropna(subset=['is_late'])
print(df.shape, '   is_late rate:', round(df['is_late'].mean(), 3))

---

## Section 2: Define column groups

Different column types need different treatment:
- **Numeric** (distance_km, log_freight) → impute with median + scale
- **Categorical** (payment_type) → impute with most-frequent + one-hot

ColumnTransformer applies different pipelines to different column groups.

In [ ]:
NUM = ['distance_km', 'log_freight', 'estimate_days', 'purchase_hour',
       'num_items', 'total_price']
CAT = ['payment_type']
TARGET = 'is_late'

X = df[NUM + CAT]
y = df[TARGET]
print('Features:', NUM + CAT)
print('Target distribution:')
print(y.value_counts(normalize=True).round(3))

---

## Section 3: Build the ColumnTransformer

Two mini-pipelines (one per column type), bundled into a ColumnTransformer.

In [ ]:
# Pipeline for numeric columns: median impute + standard scale
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# Pipeline for categorical columns: most-frequent impute + one-hot
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, NUM),
    ('cat', categorical_pipe, CAT),
])

print('✅ ColumnTransformer built. It applies:')
print(f'   numeric_pipe to:    {NUM}')
print(f'   categorical_pipe to: {CAT}')

---

## Section 4: Wrap with a model and train end-to-end

### 🚨 Imbalance is real

Olist `is_late` is 7% positive. A model that always predicts "on time":
- ✅ 93% accuracy
- ❌ Catches 0 late orders

Useless! We use `class_weight='balanced'` to make the model care about the rare class.

In [ ]:
# Full pipeline: preprocessing + model
full_pipe = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(class_weight='balanced',
                               max_iter=1000, random_state=42)),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

full_pipe.fit(X_train, y_train)

y_pred = full_pipe.predict(X_test)
y_proba = full_pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f'\nROC-AUC: {roc_auc_score(y_test, y_proba):.3f}')

### 💡 Reading the classification report

Look at the row for **class 1 (late)**:
- **precision** ≈ 0.20 — "of the orders we flagged late, 20% really were"
- **recall** ≈ 0.65 — "we caught 65% of all late orders"
- **f1** ≈ 0.30 — balanced score

We catch ⅔ of late orders! Without `class_weight='balanced'`, recall would be near zero. **The imbalance handling matters more than the model choice.**

---

## Section 5: Save the pipeline 💾

**One file = production-ready preprocessing + model.**

In [ ]:
import joblib

# Save the preprocessor (M4 will refit and use it differently)
joblib.dump(preprocessor, 'olist_preprocessor.pkl')

# Also save the full pipeline for reference
joblib.dump(full_pipe, 'olist_pipeline_v0.pkl')

print('✅ Saved 2 files:')
print('   olist_preprocessor.pkl   → just preprocessing (load in M4)')
print('   olist_pipeline_v0.pkl    → full pipeline incl. logistic baseline')
print()
print('🔗 In M8 (Capstone), the FastAPI /predict endpoint loads')
print('   olist_preprocessor.pkl + the trained model from M4-M7.')

---

## Section 6: A note on imbalance handling 🎚

We used `class_weight='balanced'`. There are other tools you'll meet:

| Tool | When to use |
| --- | --- |
| `class_weight='balanced'` | **Default. Works for most models.** Already does what you need 80% of the time. |
| `imbalanced-learn`'s **SMOTE** | Synthetic oversampling — when positives are <1% |
| **Threshold tuning** | Lower the 0.5 cutoff to e.g. 0.3 — more recall, less precision (covered in M4-4) |
| **Cost-sensitive metrics** | Use F1, PR-AUC, NOT accuracy |

**Olist `is_late` at 7%:** `class_weight='balanced'` is enough. SMOTE adds complexity for marginal gain.

**If your project has 0.1% fraud or 0.01% disease:** SMOTE is worth it.

**Always:** ditch accuracy, use F1 + PR-AUC.

---

## ✅ Quick Check

1. Why fit the scaler on training data only, not the full dataset?
2. What does `OneHotEncoder(handle_unknown='ignore')` do, and why is it important for production?
3. With `is_late` at 7%, what accuracy can a 'predict always 0' model achieve? Is it useful?

## 🗺️ Where this pipeline lands

| Module | What it does with the saved pipeline |
| --- | --- |
| **M3-6** (lab) | Combines all M3 outputs into `olist_clean.parquet` |
| **M4-1 to M4-6** | Loads `olist_preprocessor.pkl`, refits on train, ships `model_v1.pkl` |
| **M5** | Same preprocessor, different downstream model (clustering) |
| **M6** | NN expects scaled inputs — same preprocessor |
| **M8** | FastAPI `/predict` endpoint loads the preprocessor + model + applies to incoming JSON |

Today's pipeline = the bridge between research code and production code.

## 📤 Submit

### Bronze
Run the full pipeline. Submit `olist_preprocessor.pkl`.

### Gold
1. Try SMOTE oversampling (`pip install imbalanced-learn`). Compare F1 vs `class_weight='balanced'`.
2. Half-page rationale: which imbalance handling would you choose for Olist and why?

🚧 *Great work — your code is now production-ready!*